# Reinforcement Learning Fine-Tuning with GRPO — Colab

This notebook mirrors `docs/chapter_llm_ft/grpo.md`. You will:
- Set up a small LLM with quantization and optional LoRA
- Prepare a tiny synthetic dataset with references
- Implement simple reward functions (format + numeric tolerance)
- Run `trl.GRPOTrainer` with a small configuration suitable for Colab

We use a lightweight model so sampling multiple generations is affordable.


In [ ]:
# Install libraries for GRPO demos
import sys, subprocess

def pip_install(packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *packages])

pip_install([
    "transformers",
    "datasets",
    "accelerate",
    "trl",
    "peft",
    "bitsandbytes",
])

import torch, transformers, datasets, accelerate, trl, peft
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)


## Model and tokenizer (small model for Colab)

We’ll use TinyLlama with 4-bit loading. For Colab’s limits, we’ll keep batch/generations small.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
)

print("Loaded:", model_id)


## Optional: add LoRA

If you hit OOM or want faster training, add LoRA adapters.


In [ ]:
from peft import LoraConfig, get_peft_model

try:
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        target_modules="all-linear",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    # Align adapter dtype with desired compute dtype (float16 on T4)
    try:
        model = model.to(compute_dtype)
    except Exception:
        pass
    model.print_trainable_parameters()
except Exception as e:
    print("Skipping LoRA:", e)


## Tiny synthetic dataset and references

To keep it runnable on Colab, we’ll build a tiny list of prompts with ground-truth numeric answers. In practice, replace with your dataset (e.g., MedCalc-Bench).


In [ ]:
SYSTEM_PROMPT = (
    "You are a clinical calculator assistant. "
    "Provide concise reasoning in <think>...</think> and the final numeric result in <answer>...</answer>."
)

def build_prompt(patient_note, question):
    return (
        f"{SYSTEM_PROMPT}\n\n"
        f"Patient Note: {patient_note}\n"
        f"Question: {question}\n"
        f"Answer with <think> and <answer> tags."
    )

train_samples = [
    {"prompt": build_prompt("BMI example", "Height 1.75m, Weight 70kg."), "reference": "22.86"},
    {"prompt": build_prompt("Creatinine clearance", "Compute Cockcroft-Gault."), "reference": "95"},
]


## Reward functions (format + numeric tolerance)

We start with two simple, deterministic rewards:
- Format reward: output must contain both `<think>...</think>` and `<answer>...</answer>`
- Numeric tolerance reward: final `<answer>` matches reference within absolute tolerance


In [ ]:
import re

def format_reward(completions, **kwargs):
    pattern = re.compile(r"<think>.*?</think>\s*<answer>.*?</answer>", re.DOTALL)
    return [1.0 if isinstance(c, str) and pattern.search(c) else 0.0 for c in completions]


def numeric_tolerance_reward(completions, references=None, atol=0.5, **kwargs):
    out = []
    for c, ref in zip(completions, references or []):
        try:
            pred_m = re.search(r"<answer>\s*([+-]?[0-9]*\.?[0-9]+)", c)
            ref_m = re.search(r"([+-]?[0-9]*\.?[0-9]+)", ref)
            if pred_m and ref_m:
                pred_v = float(pred_m.group(1))
                ref_v = float(ref_m.group(1))
                out.append(1.0 if abs(pred_v - ref_v) <= atol else 0.0)
            else:
                out.append(0.0)
        except Exception:
            out.append(0.0)
    return out


def reward_wrapper(func):
    def _wrapped(completions, samples, **kwargs):
        refs = [s.get("reference", "") for s in samples]
        return func(completions, references=refs, **kwargs)
    return _wrapped


## GRPO config and trainer (small settings)

We keep generations and sequence lengths small to fit Colab. This is for learning the loop, not for SOTA results.


In [ ]:
from trl import GRPOConfig, GRPOTrainer

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
config = GRPOConfig(
    output_dir="tinyllama-grpo-demo",
    learning_rate=1e-5,
    gradient_accumulation_steps=4,
    per_device_train_batch_size=1,
    num_train_epochs=1,
    bf16=use_bf16,
    fp16=not use_bf16,
    max_prompt_length=256,
    max_completion_length=96,
    num_generations=2,
    logging_steps=5,
    save_strategy="no",
    report_to=["none"],
    remove_unused_columns=False,
)

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    args=config,
    train_dataset=train_samples,
    reward_funcs=[format_reward, reward_wrapper(numeric_tolerance_reward)],
)

trainer.train()
trainer.save_model()
